# NYC Elevator Complaints — The Summer Spike

> **Narrative:** *"NYC elevator complaints spike 35% in July. Summer is the worst time to be in a building with a history of failures — and that window is weeks away."*

Source: [NYC Open Data — DOB Elevator Complaint Dispositions](https://data.cityofnewyork.us/Housing-Development/DOB-Elevator-Complaint-Dispositions/kqwi-7ncn)  
Codes: `6S` · `6M` · Years on record: 2018–present

In [ ]:
# ── PARAMETERS — edit before presenting ──────────────────────────
YEARS = list(range(2018, 2026))  # years to analyze (2026 is still in progress)
# To show a single year:  YEARS = [2024]
# ─────────────────────────────────────────────────────────────────

import os, sys, requests
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../../../.env"))
except ImportError:
    pass

TOKEN = os.getenv("SODA_APP_TOKEN")
if not TOKEN:
    print("⚠  SODA_APP_TOKEN not set — add it to .env or export it in your shell")

SODA_URL = "https://data.cityofnewyork.us/resource/kqwi-7ncn.json"
MONTHS   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
PREFIXES = ["01/","02/","03/","04/","05/","06/","07/","08/","09/","10/","11/","12/"]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f5f5f5",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "sans-serif",
    "axes.titlesize":   14,
    "axes.labelsize":   11,
})

print(f"Parameters: years={YEARS[0]}–{YEARS[-1]}  ({len(YEARS)} years × 12 months = {len(YEARS)*12} API calls)")

In [ ]:
def fetch_month(prefix: str, year: int) -> int:
    params = {
        "$select": "count(*) AS cnt",
        "$where": f"complaint_category IN ('6S','6M') AND date_entered LIKE '{prefix}%/{year}'",
    }
    if TOKEN:
        params["$$app_token"] = TOKEN
    r = requests.get(SODA_URL, params=params, timeout=15)
    r.raise_for_status()
    data = r.json()
    return int(data[0].get("cnt", 0)) if data else 0

all_data: dict[int, list[int]] = {}

for year in YEARS:
    sys.stdout.write(f"  {year}: ")
    sys.stdout.flush()
    monthly = []
    for prefix in PREFIXES:
        monthly.append(fetch_month(prefix, year))
        sys.stdout.write(".")
        sys.stdout.flush()
    print(f"  total={sum(monthly):,}")
    all_data[year] = monthly

print("\n✓ Data loaded")

## Average Monthly Pattern (All Years)

The summer spike is consistent across every year since 2018.

In [ ]:
# Compute per-month averages across all years
avg_by_month = []
for m in range(12):
    vals = [all_data[y][m] for y in YEARS if all_data[y][m] > 0]
    avg_by_month.append(sum(vals) / len(vals) if vals else 0)

overall_avg = sum(avg_by_month) / len(avg_by_month)

bar_colors = [
    "#e63946" if avg > overall_avg * 1.05 else
    ("#a8dadc" if avg < overall_avg * 0.95 else "#457b9d")
    for avg in avg_by_month
]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(MONTHS, avg_by_month, color=bar_colors, edgecolor="white", width=0.65)
ax.axhline(overall_avg, color="#333", linewidth=1.2, linestyle="--", alpha=0.6, label=f"Annual avg ({overall_avg:.0f})")

for bar, avg in zip(bars, avg_by_month):
    delta = (avg - overall_avg) / overall_avg * 100 if overall_avg else 0
    if abs(delta) > 5:
        sign = "+" if delta > 0 else ""
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
                f"{sign}{delta:.0f}%", ha="center", fontsize=9, fontweight="bold",
                color="#e63946" if delta > 0 else "#457b9d")

ax.set_ylabel("Avg complaints / month")
ax.set_title(f"Average Monthly Elevator Complaints — NYC ({YEARS[0]}–{YEARS[-1]})", pad=12)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
fig.tight_layout()
plt.show()

jul_idx = 6
jul_pct = (avg_by_month[jul_idx] - overall_avg) / overall_avg * 100
print(f"July average: {avg_by_month[jul_idx]:.0f} complaints  ({jul_pct:+.1f}% above annual mean)")

## Year-Over-Year Heatmap

Every row is one year. Every column is one month. Darker red = more complaints.

In [ ]:
matrix = np.array([all_data[y] for y in YEARS], dtype=float)

fig, ax = plt.subplots(figsize=(11, len(YEARS) * 0.55 + 1.5))
im = ax.imshow(matrix, aspect="auto", cmap="YlOrRd")

ax.set_xticks(range(12))
ax.set_xticklabels(MONTHS)
ax.set_yticks(range(len(YEARS)))
ax.set_yticklabels(YEARS)
ax.set_title("Elevator Complaints by Month & Year — NYC", pad=12)

# Annotate cells
for i, year in enumerate(YEARS):
    for j, val in enumerate(all_data[year]):
        ax.text(j, i, str(val), ha="center", va="center",
                fontsize=8, color="white" if val > matrix.max() * 0.6 else "#333")

plt.colorbar(im, ax=ax, label="Complaints", shrink=0.8)
fig.tight_layout()
plt.show()

## The Headline Number

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.axis("off")
ax.text(0.5, 0.68, f"+{jul_pct:.0f}%",
        ha="center", va="center", fontsize=64, fontweight="bold",
        color="#e63946", transform=ax.transAxes)
ax.text(0.5, 0.18,
        f"more elevator complaints in July vs. the annual monthly average\n"
        f"(consistent across every year from {YEARS[0]}–{YEARS[-1]})",
        ha="center", va="center", fontsize=12, color="#555",
        transform=ax.transAxes)
fig.tight_layout()
plt.show()